# 06 — Market-Aware Price Prediction
Task 7: move from “what will the price be” to “what price can I expect in each market, and which market looks better.”

## 1. Load the feature dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv(
    "../data/processed/tomato_features.csv"
)

df["date"] = pd.to_datetime(df["date"])

print("Shape:", df.shape)
df.head()

## 2. Check markets

In [ ]:
print("Number of markets:", df["market"].nunique())

print("\nMarkets:")
print(df["market"].unique())

In [ ]:
market_counts = df["market"].value_counts()

print(market_counts)

In [ ]:
plt.figure(figsize=(10, 5))

market_counts.head(15).plot(
    kind="bar"
)

plt.xlabel("Market")
plt.ylabel("Number of Records")
plt.title("Records by Market")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. Chronological ordering
Essential because the historical features (lags, rolling stats) are market-specific.

In [ ]:
df = df.sort_values(
    ["market", "date"]
).reset_index(drop=True)

## 4. Create market encoding
XGBoost needs numerical inputs, so markets are one-hot encoded rather than passed as raw text.

In [ ]:
market_dummies = pd.get_dummies(
    df["market"],
    prefix="market",
    dtype=int
)

market_dummies.head()

In [ ]:
df = pd.concat(
    [df, market_dummies],
    axis=1
)

## 5. Define base features
Same feature set as Task 5/6.

In [ ]:
BASE_FEATURES = [
    "min_price",
    "max_price",
    "average_price",
    "price_range",

    "year",
    "month",
    "day",
    "day_of_week",

    "month_sin",
    "month_cos",

    "price_lag_1",
    "price_lag_7",
    "price_lag_14",
    "price_lag_30",

    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_30",

    "rolling_std_7",

    "price_change_1",
    "price_change_pct"
]

if "arrival_quantity" in df.columns:

    BASE_FEATURES += [
        "arrival_quantity",
        "arrival_lag_1",
        "arrival_rolling_7"
    ]

## 6. Add market features

In [ ]:
MARKET_FEATURES = list(
    market_dummies.columns
)

FEATURES = (
    BASE_FEATURES
    + MARKET_FEATURES
)

print("Total features:", len(FEATURES))

## 7. Create the modeling dataset

In [ ]:
TARGET = "modal_price"

model_df = df[
    FEATURES
    + [
        TARGET,
        "date",
        "market"
    ]
].copy()

In [ ]:
model_df = model_df.replace(
    [np.inf, -np.inf],
    np.nan
)

In [ ]:
model_df = model_df.dropna(
    subset=FEATURES + [TARGET]
)

print("Final modeling data:", model_df.shape)

## 8. Time-based split
No random split — chronological only.

In [ ]:
model_df = model_df.sort_values(
    "date"
).reset_index(drop=True)

split_index = int(
    len(model_df) * 0.8
)

train = model_df.iloc[
    :split_index
]

test = model_df.iloc[
    split_index:
]

In [ ]:
print(
    "Training period:",
    train["date"].min(),
    "→",
    train["date"].max()
)

print(
    "Testing period:",
    test["date"].min(),
    "→",
    test["date"].max()
)

## 9. Create X and y

In [ ]:
X_train = train[FEATURES]
y_train = train[TARGET]

X_test = test[FEATURES]
y_test = test[TARGET]

## 10. Train market-aware XGBoost

In [ ]:
from xgboost import XGBRegressor

market_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

In [ ]:
market_model.fit(
    X_train,
    y_train
)

## 11. Predict

In [ ]:
market_predictions = market_model.predict(
    X_test
)

## 12. Evaluate

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

market_mae = mean_absolute_error(
    y_test,
    market_predictions
)

market_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        market_predictions
    )
)

market_mape = np.mean(
    np.abs(
        (y_test - market_predictions)
        / y_test
    )
) * 100

market_r2 = r2_score(
    y_test,
    market_predictions
)

print("Market-Aware XGBoost")
print("--------------------")
print(f"MAE  : {market_mae:.2f}")
print(f"RMSE : {market_rmse:.2f}")
print(f"MAPE : {market_mape:.2f}%")
print(f"R²   : {market_r2:.4f}")

## 13. Create prediction table
The important part for Kadak Bhav — per-record predictions tagged by market.

In [ ]:
market_results = pd.DataFrame({
    "date": test["date"].values,
    "market": test["market"].values,
    "actual_price": y_test.values,
    "predicted_price": market_predictions
})

market_results["absolute_error"] = np.abs(
    market_results["actual_price"]
    - market_results["predicted_price"]
)

market_results.head(10)

## 14. Compare market performance

In [ ]:
market_performance = (
    market_results
    .groupby("market")
    .agg(
        records=("actual_price", "count"),
        actual_avg=("actual_price", "mean"),
        predicted_avg=("predicted_price", "mean"),
        avg_error=("absolute_error", "mean")
    )
    .reset_index()
)

market_performance.sort_values(
    "predicted_avg",
    ascending=False
).head(10)

## 15. Build the “best market” logic
A high predicted price alone isn't "best" — Task 8 adds transport cost, buyer reliability, and perishability risk on top of this to get Net Realization. This is just the price-ranking piece.

In [ ]:
best_markets = (
    market_performance
    .sort_values(
        "predicted_avg",
        ascending=False
    )
)

best_markets.head(10)

## 16. Create a simple recommendation function
Price-based prototype only — not the final Net Realization recommendation.

In [ ]:
def recommend_market(
    market_table,
    top_n=5
):

    recommendations = (
        market_table
        .sort_values(
            "predicted_avg",
            ascending=False
        )
        .head(top_n)
        .copy()
    )

    recommendations[
        "rank"
    ] = range(
        1,
        len(recommendations) + 1
    )

    return recommendations

In [ ]:
recommendations = recommend_market(
    market_performance
)

recommendations

## 17. Visualize market predictions

In [ ]:
top_markets = (
    market_performance
    .sort_values(
        "predicted_avg",
        ascending=False
    )
    .head(10)
)

In [ ]:
plt.figure(figsize=(10, 5))

plt.bar(
    top_markets["market"],
    top_markets["predicted_avg"]
)

plt.xlabel("Market")
plt.ylabel("Predicted Modal Price")
plt.title(
    "Top Markets by Predicted Price"
)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 18. Feature importance
Let the model tell you whether market identity, lags, or arrivals matter most — don't assume beforehand.

In [ ]:
importance = pd.Series(
    market_model.feature_importances_,
    index=FEATURES
).sort_values(
    ascending=False
)

In [ ]:
plt.figure(figsize=(10, 6))

importance.head(20).sort_values().plot(
    kind="barh"
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title(
    "Market-Aware XGBoost Feature Importance"
)

plt.tight_layout()
plt.show()

## 19. Save the results

In [ ]:
market_results.to_csv(
    "../data/processed/market_predictions.csv",
    index=False
)

In [ ]:
market_performance.to_csv(
    "../data/processed/market_performance.csv",
    index=False
)

In [ ]:
recommendations.to_csv(
    "../data/processed/market_recommendations.csv",
    index=False
)

## 20. Save the model

In [ ]:
import joblib
import os

os.makedirs(
    "../models",
    exist_ok=True
)

joblib.dump(
    market_model,
    "../models/market_aware_xgboost.pkl"
)

print(
    "Market-aware model saved."
)

## Next step
Task 8 builds the Net Realization Engine on top of this: predicted price − transport cost − storage cost − expected perishability loss, combined with buyer reliability, to turn “highest predicted price” into “highest expected net realization.”